# Create video sequence of AMTM band intensities

In [1]:
import sys
sys.path.append('../scripts')
import utils_3d as three
sys.path.append('/home/r/Robert.Reichert/juwavelet')
from juwavelet import parallel
import xarray as xr
import numpy as np
import pylops
import matplotlib.pyplot as plt

plt.style.use("../latex_default.mplstyle")

In [2]:
orig_data_path = '/project/meteo/work/Robert.Reichert/data/AMTM/Rio_Grande/2018/201805/AMTM_BI_20180516-2137.nc'
domi_data_path = '../data/AMTM_dominant_wave_mode_recon.nc'
ds_orig = xr.open_dataset(orig_data_path)
ds_domi = xr.open_dataset(domi_data_path)

/home/r/Robert.Reichert/.conda/envs/JuWavelet/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [3]:
BI_orig = ds_orig['band intensity'].isel(time=slice(100,250),x=slice(3,-40),y=slice(35,-42))
BI_high = BI_orig.values.copy()
BI_high = np.flip(BI_high,axis=0)
BI_high, BI_low = three.BG_removal(BI_high,max_order=1,fourier_radius=1)

In [4]:
dx, dy, dt = 625, 625, 35
nx, ny, nt = BI_orig['y'].size, BI_orig['x'].size, BI_orig['time'].size

nmask   = (ny, nx, nt)
ntap    = (20,20,5)
tap3d   = pylops.utils.tapers.tapernd(nmask, ntap, "hanning")
BI_work = BI_high*tap3d

In [5]:
a  = 2*dx/dt
s0 = 4 #*min(dx,dy,dt*a)
s1 = 0.5 * min([nx,ny,nt])
dj = 1/4
#js = int(1/dj*np.log2(np.sqrt(nx**2+ny**2+nt**2)*max(dx,dy,dt*a)/s0))
js = int(1/dj*np.log2(s1/s0))
#scales = s0 * (2 ** (np.arange(js) * dj))
jt=10
jp=15
print(a, nx, ny, nt, js, jt, jp, js*jt*jp)

35.714285714285715 243 213 150 16 10 15 2400


In [6]:
noise_floor = 0.1*np.std(BI_work)
print('Noise floor:', noise_floor)
cwt_result = parallel.decompose3d_parallel(np.transpose(BI_work,(1,0,2)), dx, dy, dt, s0*min([dx,dy,dt*a]), dj, js, jt, jp, aspect=a,
    nxpad=None, nypad=None, nzpad=None, opts={'param': 2*np.pi},mode="scaled", dtype=np.complex64, filt={'min_coefficient': noise_floor})

Noise floor: 169.26246510134354
pr/tr 4 4


  0%|          | 0/16 [00:00<?, ?it/s]/home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331: RuntimeWarning: divide by zero encountered in divide
  wxs = periods[:, np.newaxis, np.newaxis] / (
/home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331: RuntimeWarning: divide by zero encountered in divide
  wxs = periods[:, np.newaxis, np.newaxis] / (
/home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331: RuntimeWarning: divide by zero encountered in divide
  wxs = periods[:, np.newaxis, np.newaxis] / (
/home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331: RuntimeWarning: divide by zero encountered in divide
  wxs = periods[:, np.newaxis, np.newaxis] / (
  6%|▋         | 1/16 [09:23<2:20:53, 563.58s/it]/home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331: RuntimeWarning: divide by zero encountered in divide
  wxs = periods[:, np.newaxis, np.newaxis] / (
 12%|█▎        | 2/16 [09:39<56:22, 241.57s/it]  /home/r/Robert.Reichert/juwavelet/juwavelet/parallel.py:331

In [38]:
reduced_WAS  = three.get_reduced_WAS(cwt_result['decomposition'])
denoised_WAS = three.denoise_reduced_WAS(reduced_WAS, white_noise_level=noise_floor, sMAD_threshold=3)

In [40]:
cwt_result['decomposition'] = three.filter_coefficients_3d(cwt_result['decomposition'], denoised_WAS)

In [7]:
wavy_stuff = parallel.reconstruct3d_parallel(cwt_result)

100%|██████████| 16/16 [02:28<00:00,  9.30s/it]


In [8]:
noisy_stuff = np.transpose(BI_work,(1,0,2)) - wavy_stuff

In [9]:
BI_min, BI_max = np.percentile(BI_orig,[1,99])
BI_lev = np.linspace(BI_min,BI_max,51)
BI_ticks = [24_000,26_000,28_000,30_000,32_000,34_000,36_000]
BI_pert_ticks = [-1_500,-1_000,-500,0,500,1_000,1_500]
BI_pert = np.linspace(-1_500,1_500,51)
BI_pert_2 = np.linspace(-3_000,3_000,51)

x=BI_orig.x#np.linspace(0,(nx-1)*dx,nx)*1e-3
y=BI_orig.y#np.linspace(0,(ny-1)*dy,ny)*1e-3
gskw = {"hspace": 0.05, "wspace": 0.05, "width_ratios": [1, 1], "height_ratios": [1, 1]}

dpi = 150
figsize = (1920/dpi, 1080/dpi)

for time_idx in range(150):
#time_idx=115

    fig, ax=plt.subplots(2,2, figsize=figsize, gridspec_kw=gskw, sharex=True, sharey=True)
    #ax[0,0].axis('off')
    
    plot_original = ax[0,0].contourf(y,x,np.flip(BI_orig.isel(time=time_idx),axis=0),levels=BI_lev,cmap='bone',extend='both')
    cbar = fig.colorbar(plot_original, ticks=BI_ticks, label='BI')#, ax=ax[0,0], location='top', ticks=[-4000,-2000,0,2000,4000], fraction=1, shrink=0.9, extend='both',label='Band intensity', aspect=25)
    
    plot_wavefield = ax[0,1].contourf(y,x,wavy_stuff[:,:,time_idx].T,levels=BI_pert,cmap='RdBu_r',extend='both')
    cbar = fig.colorbar(plot_wavefield, ticks=BI_pert_ticks, label='BI')
    
    plot_background = ax[1,1].contourf(y,x,BI_low[:,:,time_idx]+np.mean(BI_orig.values)+noisy_stuff[:,:,time_idx].T,levels=BI_lev,cmap='bone',extend='both')
    cbar = fig.colorbar(plot_background, ticks=BI_ticks, label='BI')
    
    plot_dominant = ax[1,0].contourf(y,x,ds_domi['dominant_mode_reconstruction'].isel(time=time_idx).T,levels=BI_pert,cmap='RdBu_r',extend='both')
    cbar = fig.colorbar(plot_dominant, ticks=BI_pert_ticks, label='BI')
    
    ax[0,0].set_ylabel('Meridional distance / km')
    ax[1,0].set_ylabel('Meridional distance / km')
    ax[1,0].set_xlabel('Zonal distance / km')
    ax[1,1].set_xlabel('Zonal distance / km')
    
    plt.savefig(f'./figures/AMTM_supp/AMTM_supp_fig_{time_idx:03d}.png', dpi=dpi)
    plt.close()

In [10]:
import os
import moviepy.video.io.ImageSequenceClip

image_folder = './figures/AMTM_supp/'
fps = 15

image_files = [
    os.path.join(image_folder, img)
    for img in os.listdir(image_folder)
    if img.endswith(".png")
]
image_files.sort()
clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
clip.write_videofile("./animations/AMTM_supplement_v3.mp4")

Moviepy - Building video ./animations/AMTM_supplement_v3.mp4.
Moviepy - Writing video ./animations/AMTM_supplement_v3.mp4



Moviepy - Done !
Moviepy - video ready ./animations/AMTM_supplement_v3.mp4
